# Cálculo de variables objetivo - TIBAITATA

Este notebook preparará las variables necesarias para el cálculo de evapotranspiración de referencia mediante FAO-56.

## Importaciones

In [22]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuración de visualización

Se muestran todas las columnas de los DataFrames y se establece una precisión numérica uniforme para facilitar la inspección de resultados.

In [23]:
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
plt.style.use("seaborn-v0_8-whitegrid")

## Rutas de datos

Se declaran las rutas y se cargan los datasets sin aplicar limpieza ni preprocesamiento.

In [24]:
DATASET_LIMPIO_PATH = "./Datasets/TibaitataLimpio.pkl"
DATASET_ESTACION_PATH = "./Datasets/tibaitata.csv"

df_tibaitata_limpio = pd.read_pickle(DATASET_LIMPIO_PATH)
df_tibaitata_crudo = pd.read_csv(DATASET_ESTACION_PATH)

print("Dataset limpio:", df_tibaitata_limpio.shape)
print("Estacion:", df_tibaitata_crudo.shape)

Dataset limpio: (14545, 9)
Estacion: (1, 20)


## Constantes físicas para FAO-56

- `SOLAR_CONSTANT`: radiación extraterrestre solar, expresada en MJ m⁻² min⁻¹.
- `STEFAN_BOLTZMANN`: constante de Stefan-Boltzmann para la radiación neta de onda larga, en MJ K⁻⁴ m⁻² día⁻¹.
- `LATENT_HEAT_VAPORIZATION`: energía necesaria para vaporizar un kilogramo de agua, en MJ kg⁻¹.
- `SPECIFIC_HEAT_AIR`: calor específico del aire a presión constante, en MJ kg⁻¹ °C⁻¹.
- `EPSILON`: relación entre las masas molares del vapor de agua y el aire seco, adimensional.
- `KELVIN_OFFSET`: desplazamiento para convertir temperaturas de °C a K.
- `SECONDS_PER_DAY`: número de segundos de un día, útil para conversiones de tiempo.


In [25]:
SOLAR_CONSTANT = 0.0820  # MJ m^-2 min^-1
STEFAN_BOLTZMANN = 4.903e-9  # MJ K^-4 m^-2 day^-1
LATENT_HEAT_VAPORIZATION = 2.45  # MJ kg^-1
SPECIFIC_HEAT_AIR = 1.013e-3  # MJ kg^-1 °C^-1
EPSILON = 0.622  # adimensional
KELVIN_OFFSET = 273.15  # K por °C
SECONDS_PER_DAY = 86_400  # s day^-1

## Organización de variables meteorológicas

Se transforma el dataset limpio de formato largo a formato ancho. Antes del pivotado se revisan las claves duplicadas `Fecha`-`parametro_limpio`; si existen, el proceso se detiene para evitar promediar o eliminar registros automáticamente.

In [26]:
parametros_et0 = {
    "Humedad relativa del aire a 2 metros media diaria": "RH_mean",
    "Radiación solar diaria": "Radiation_Wh_m2",
    "Temperatura máxima diaria": "Tmax",
    "Temperatura mínima diaria": "Tmin",
    "Velocidad 10 minutal del viento media diaria": "Wind_speed",
}

datos_et0 = df_tibaitata_limpio[
    df_tibaitata_limpio["parametro_limpio"].isin(parametros_et0)
].copy()

duplicados_et0 = datos_et0[
    datos_et0.duplicated(["Fecha", "parametro_limpio"], keep=False)
].sort_values(["Fecha", "parametro_limpio"])

if duplicados_et0.empty:
    print("No se encontraron duplicados para la combinación Fecha-parametro_limpio.")
else:
    print("Se encontraron los siguientes registros duplicados:")
    display(duplicados_et0[["Fecha", "parametro_limpio", "Valor"]])
    raise ValueError(
        "No se puede realizar el pivotado: existen duplicados para "
        "la combinación Fecha-parametro_limpio."
    )

fechas_disponibles = df_tibaitata_limpio[["Fecha"]].drop_duplicates()

df_tibaitata_et0 = (
    datos_et0.assign(parametro_limpio=datos_et0["parametro_limpio"].map(parametros_et0))
    .pivot(index="Fecha", columns="parametro_limpio", values="Valor")
    .reindex(columns=parametros_et0.values())
    .rename_axis(None, axis=1)
    .reset_index()
)

df_tibaitata_et0 = (
    fechas_disponibles.merge(df_tibaitata_et0, on="Fecha", how="left")
    .sort_values("Fecha")
    .reset_index(drop=True)
)

print("Primeras filas:")
display(df_tibaitata_et0.head())
print("Dimensiones:", df_tibaitata_et0.shape)
print("Tipos de datos:")
display(df_tibaitata_et0.dtypes)
print("Valores nulos por columna:")
display(df_tibaitata_et0.isna().sum())

No se encontraron duplicados para la combinación Fecha-parametro_limpio.
Primeras filas:


,Fecha,RH_mean,Radiation_Wh_m2,Tmax,Tmin,Wind_speed
0,2005-02-02,89.531,1696.5,19.8,4.6,1.645
1,2005-02-03,83.958,4480.5,20.2,5.6,1.965
2,2005-02-04,89.237,2363.3,18.6,6.8,2.149
3,2005-02-05,80.562,3137.0,20.4,9.0,2.558
4,2005-02-06,89.500,1992.7,18.8,10.8,1.965


Dimensiones: (2909, 6)
Tipos de datos:


Fecha              datetime64[us]
RH_mean                   float64
Radiation_Wh_m2           float64
Tmax                      float64
Tmin                      float64
Wind_speed                float64
dtype: object

Valores nulos por columna:


Fecha              0
RH_mean            0
Radiation_Wh_m2    0
Tmax               0
Tmin               0
Wind_speed         0
dtype: int64

In [27]:
columnas_originales_et0 = df_tibaitata_et0.columns.tolist()
numero_registros_et0 = len(df_tibaitata_et0)

# El cálculo vectorizado conserva NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["T_mean"] = (df_tibaitata_et0["Tmax"] + df_tibaitata_et0["Tmin"]) / 2

assert len(df_tibaitata_et0) == numero_registros_et0
assert df_tibaitata_et0.columns[:len(columnas_originales_et0)].tolist() == columnas_originales_et0
assert df_tibaitata_et0["T_mean"].isna().equals(
    df_tibaitata_et0[["Tmax", "Tmin"]].isna().any(axis=1)
)
assert (
    df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "T_mean"]
    == (
        df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmax"]
        + df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmin"]
    ) / 2
).all()

print("Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.")
display(df_tibaitata_et0[["Tmax", "Tmin", "T_mean"]].head())
print("Valores nulos en T_mean:", df_tibaitata_et0["T_mean"].isna().sum())

Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.


,Tmax,Tmin,T_mean
0,19.8,4.6,12.2
1,20.2,5.6,12.9
2,18.6,6.8,12.7
3,20.4,9.0,14.7
4,18.8,10.8,14.8


Valores nulos en T_mean: 0


In [28]:
# Presión de vapor de saturación según FAO-56.
def presion_vapor_saturacion(temperatura):
    return 0.6108 * np.exp((17.27 * temperatura) / (temperatura + 237.3))

columnas_antes_es = df_tibaitata_et0.columns.tolist()
valores_antes_es = df_tibaitata_et0[columnas_antes_es].copy()

# Las operaciones vectorizadas conservan NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["es_tmax"] = presion_vapor_saturacion(df_tibaitata_et0["Tmax"])
df_tibaitata_et0["es_tmin"] = presion_vapor_saturacion(df_tibaitata_et0["Tmin"])
df_tibaitata_et0["es"] = (df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2

assert df_tibaitata_et0[columnas_antes_es].equals(valores_antes_es)
assert df_tibaitata_et0["es_tmax"].isna().equals(df_tibaitata_et0["Tmax"].isna())
assert df_tibaitata_et0["es_tmin"].isna().equals(df_tibaitata_et0["Tmin"].isna())
assert np.isfinite(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna()).all().all()
assert (df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna() > 0).all().all()
assert np.allclose(
    df_tibaitata_et0["es"].dropna(),
    ((df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2).dropna(),
)

print("Resultados de presión de vapor de saturación (kPa):")
display(df_tibaitata_et0[["Tmax", "Tmin", "es_tmax", "es_tmin", "es"]].head())
print("Valores nulos en las columnas nuevas:")
display(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].isna().sum())

Resultados de presión de vapor de saturación (kPa):


,Tmax,Tmin,es_tmax,es_tmin,es
0,19.8,4.6,2.309,0.848,1.579
1,20.2,5.6,2.367,0.910,1.638
2,18.6,6.8,2.143,0.988,1.566
3,20.4,9.0,2.397,1.148,1.772
4,18.8,10.8,2.170,1.295,1.733


Valores nulos en las columnas nuevas:


es_tmax    0
es_tmin    0
es         0
dtype: int64

## Presión real de vapor

Se calcula `ea` mediante la aproximación FAO-56 basada en la humedad relativa media diaria: `ea = es * RH_mean / 100`. La humedad relativa debe estar expresada como porcentaje.

In [29]:
columnas_antes_ea = df_tibaitata_et0.columns.tolist()

assert {"es", "RH_mean"}.issubset(df_tibaitata_et0.columns)
rh_valida = df_tibaitata_et0["RH_mean"].dropna()
assert rh_valida.between(0, 100).all(), "RH_mean debe estar entre 0 y 100 %."
assert rh_valida.max() > 1, "RH_mean parece estar en escala 0-1, no en porcentaje."

df_tibaitata_et0["ea"] = (
    df_tibaitata_et0["es"] * df_tibaitata_et0["RH_mean"] / 100
)

filas_completas_ea = df_tibaitata_et0[["es", "RH_mean"]].notna().all(axis=1)
assert df_tibaitata_et0["ea"].isna().equals(~filas_completas_ea)
assert df_tibaitata_et0.columns[:len(columnas_antes_ea)].tolist() == columnas_antes_ea
assert np.isfinite(df_tibaitata_et0.loc[filas_completas_ea, "ea"]).all()
assert (
    (df_tibaitata_et0.loc[filas_completas_ea, "ea"] >= 0)
    & (df_tibaitata_et0.loc[filas_completas_ea, "ea"] <= df_tibaitata_et0.loc[filas_completas_ea, "es"])
).all()
assert np.allclose(
    df_tibaitata_et0.loc[filas_completas_ea, "ea"],
    df_tibaitata_et0.loc[filas_completas_ea, "es"]
    * df_tibaitata_et0.loc[filas_completas_ea, "RH_mean"]
    / 100,
)

print("Resultados de presión real de vapor (kPa):")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].isna().sum())
print("Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.")

Resultados de presión real de vapor (kPa):


,RH_mean,es,ea
0,89.531,1.579,1.414
1,83.958,1.638,1.376
2,89.237,1.566,1.397
3,80.562,1.772,1.428
4,89.500,1.733,1.551


Valores nulos:


RH_mean    0
es         0
ea         0
dtype: int64

Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.
